# ZeroMiss — the validation story

This notebook reproduces, from the live engine, the claims the project rests on:
the defining zero-miss property, the **Zarchan step-maneuver curve**, the APN–beats–TPN
result, the RK4 order of accuracy, and a Monte-Carlo P_k. Everything below is computed
by `zeromiss`, not hand-drawn.

```bash
pip install -e ".[dev,data,media]"   # from the repo root
```

In [ ]:
import matplotlib.pyplot as plt
from zeromiss import Engagement, scenarios, validation as V
from zeromiss.scenario import Scenario
plt.rcParams['figure.facecolor'] = '#070b14'
plt.rcParams['axes.facecolor'] = '#0c1322'
plt.rcParams['text.color'] = '#f4f8ff'
plt.rcParams['axes.labelcolor'] = '#7c8aa5'
plt.rcParams['xtick.color'] = '#7c8aa5'
plt.rcParams['ytick.color'] = '#7c8aa5'

## 1. The whole validation suite, as a pass table

In [ ]:
for c in V.run_all():
    print(('PASS' if c.passed else 'FAIL'), f'[{c.category:10s}]', c.name, '::', c.detail)

## 2. The headline: the Zarchan step-maneuver miss curve

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for N, c in {3.0:'#37e0e6', 4.0:'#ffb454', 5.0:'#f4f8ff'}.items():
    pts = V.zarchan_step_curve(N)
    ax.plot([p.t_over_tau for p in pts], [p.norm_miss for p in pts], 'o-', color=c, label=f'N={N:g}')
ax.set_xlabel('normalized flight time t_F / tau'); ax.set_ylabel('miss / (n_T tau^2)')
ax.set_title('Zarchan step-maneuver miss curve'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 3. A single engagement, end to end

In [ ]:
r = Engagement.from_yaml(scenarios.path('the_weave')).run(seed=1337)
print(r.summary())
t = r.telemetry
fig, axs = plt.subplots(1, 3, figsize=(14,3.5))
axs[0].plot(t.column('x_m'), t.column('y_m'), color='#37e0e6'); axs[0].plot(t.column('x_t'), t.column('y_t'), color='#ffb454'); axs[0].set_title('trajectories'); axs[0].set_aspect('equal')
axs[1].plot(t.column('t'), [v*57.3 for v in t.column('lambda_dot')], color='#ffb454'); axs[1].axhline(0, color='#f4f8ff', lw=0.6); axs[1].set_title('LOS rate [deg/s]')
axs[2].plot(t.column('t'), t.column('g_ach'), color='#37e0e6'); axs[2].set_title('achieved g')
plt.show()

## 4. Monte-Carlo P_k and the evasion frontier

In [ ]:
from zeromiss.montecarlo import run_campaign, sweep_pk
sc = Scenario.from_yaml(scenarios.path('the_weave'))
camp = run_campaign(sc, runs=5000, seed=0)
print(camp.summary())
sweep = sweep_pk(sc, 'N', [2,3,4,5,6], runs=2000)
fig, ax = plt.subplots(figsize=(6,4))
ax.plot([n for n,_ in sweep], [c.Pk for _,c in sweep], 'o-', color='#37e0e6')
ax.set_xlabel('N'); ax.set_ylabel('P_k'); ax.set_ylim(0,1.02); ax.set_title('P_k vs N'); ax.grid(alpha=0.3)
plt.show()

## 5. Cross-engine verification

The same scenarios are exported as fixtures (`zeromiss export-fixtures`) and replayed by
the TypeScript twin in CI, which must reproduce every run to **< 0.1%**. Independent
re-implementation agreeing to tolerance is the project's strongest verification claim.